# Regularized Linear Models (규제 선형 회귀)

앞서서 다항회귀를 사용하면 선형회귀보다 더 복잡한 패턴을 표현할 수 있음을 확인했다.

하지만 특성이 많아지고 차수가 높아질수록 모델은 학습 데이터에 지나치게 맞춰질 수 있다. 이것이 과적합이다.

규제 선형 회귀는 손실을 줄이는 것뿐 아니라 회귀계수의 크기도 함께 제어하여 모델이 과도하게 복잡해지는 것을 막는 방법이다.

> 목표는 단순히 오차가 가장 작은 모델이 아니라 오차도 작고 지나치게 흔들리지도 않는 모델을 만드는 것이다.

정규화가 적용된 비용함수는 기본 손실 함수에 계수 크기에 대한 패널티를 더한 형태로 볼 수 있다.

$
\text{목표} = \text{오차} + \text{규제항}
$

여기서 alpha는 규제의 강도를 조절하는 하이퍼파라미터이다.

- alpha가 크면 규제가 강해진다.
- alpha가 작으면 원래 선형회귀에 가까워진다.

In [40]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [41]:
# 데이터 로드
boston_housing_df = pd.read_csv('data/boston_housing_train.csv')
boston_housing_df.head()

,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
0,0.00632,18.0,2.31,0.0,0.538,6.575,65.2,4.0900,1.0,296.0,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0.0,0.469,6.421,78.9,4.9671,2.0,242.0,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0.0,0.469,7.185,61.1,4.9671,2.0,242.0,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0.0,0.458,6.998,45.8,6.0622,3.0,222.0,18.7,394.63,2.94,33.4
4,0.06905,0.0,2.18,0.0,0.458,7.147,54.2,6.0622,3.0,222.0,18.7,396.90,5.33,36.2


## L2 Ridge Regression

Ridge는 L2 규제를 사용하는 선형회귀이다.

모든 특성의 회귀계수를 완전히 없애기보다는 전체적으로 부드럽게 줄이는 방식이다.

- 다항 특성으로 복잡해진 모델의 계수 폭주를 막고
- 과적합을 완화하며
- 비교적 안정적인 일반화 성능을 기대할 수 있다.

In [42]:
# 데이터 분리
from sklearn.model_selection import train_test_split

# 타겟 컬럼
target_col = 'MEDV'
X = boston_housing_df.drop(columns=[target_col])
y = boston_housing_df[target_col]

feature_names = X.columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

(404, 13) (404,)
(102, 13) (102,)


In [43]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, root_mean_squared_error

def evaluate_reg_model(model, X_train, X_test, y_train, y_test, degree=2):
    pipeline = Pipeline([
        ('poly', PolynomialFeatures(degree=degree, include_bias=False)),
        # 규제 적용 시 회귀계수의 크기를 제어. 특성마다 단위와 범위가 다르면 규제의 유불리가 달라지므로 스케일링. 
        ('scaler', StandardScaler()),
        ('model', model)
    ])

    pipeline.fit(X_train, y_train)

    train_pred = pipeline.predict(X_train)
    test_pred = pipeline.predict(X_test)

    result = {
        'pipeline': pipeline,
        'train_r2': pipeline.score(X_train, y_train),
        'test_r2': pipeline.score(X_test, y_test),
        'train_mse': mean_squared_error(y_train, train_pred),
        'test_mse': mean_squared_error(y_test, test_pred),
        'train_mae': mean_absolute_error(y_train, train_pred),
        'test_mae': mean_absolute_error(y_test, test_pred),
        'train_rmse': root_mean_squared_error(y_train, train_pred),
        'test_rmse': root_mean_squared_error(y_test, test_pred),
        # L2 규제에서 회귀 계수 제곱합이 줄어들어 모델의 복잡도가 줄어들었음을 확인하는 요소
        'coef_l2_sum': float(np.sum(pipeline.named_steps['model'].coef_ ** 2)),
        # L1 규제에서 회귀 계수 종류가 줄어들어 모델의 복잡도가 줄어들었음을 확인하는 요소
        'zero_coef_count': int(np.sum(pipeline.named_steps['model'].coef_ == 0)),
        'coef_count': int(len(pipeline.named_steps['model'].coef_)),
    }
    return result

In [44]:
# 기본 모델 vs L2 규제 모델 비교
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet

baseline_result = evaluate_reg_model(
    LinearRegression(),
    X_train, X_test, y_train, y_test
)

ridge_result = evaluate_reg_model(
    Ridge(alpha=1.0),
    X_train, X_test, y_train, y_test
)

print('baseline_result', baseline_result)
print('ridge_result', ridge_result)

baseline_result {'pipeline': Pipeline(steps=[('poly', PolynomialFeatures(include_bias=False)),
                ('scaler', StandardScaler()), ('model', LinearRegression())]), 'train_r2': 0.9409317027113501, 'test_r2': 0.8055829447972174, 'train_mse': 5.13146404408208, 'test_mse': 14.2573381689094, 'train_mae': 1.7327630259692341, 'test_mae': 2.5748356264161987, 'train_rmse': 2.2652735031519, 'test_rmse': 3.77588905675331, 'coef_l2_sum': 91036.29984930466, 'zero_coef_count': 0, 'coef_count': 104}
ridge_result {'pipeline': Pipeline(steps=[('poly', PolynomialFeatures(include_bias=False)),
                ('scaler', StandardScaler()), ('model', Ridge())]), 'train_r2': 0.9196781431789199, 'test_r2': 0.8471791435748565, 'train_mse': 6.977833104230681, 'test_mse': 11.206931547456326, 'train_mae': 1.9584402661697309, 'test_mae': 2.2083524579642564, 'train_rmse': 2.6415588398199046, 'test_rmse': 3.347675543934377, 'coef_l2_sum': 545.4378845637893, 'zero_coef_count': 0, 'coef_count': 104}


In [45]:
# alpha 하이퍼파라미터 탐색
from sklearn.model_selection import GridSearchCV

pipeline = Pipeline([
        ('poly', PolynomialFeatures(degree=2, include_bias=False)), 
        ('scaler', StandardScaler()),
        ('model', Ridge())
    ])

# Ridge의 alpha는 규제 강도를 의미하며 일반적으로 10의 배수 단위로 탐색을 한다
# 너무 작은 값은 규제가 사라져 LinearRegression과 유사하며
# 너무 큰 값은 규제가 과도해져 과소적합 발생 가능성이 있다
param_grid = {
    'model__alpha' : [0.01, 0.1, 1, 10, 100]
}

grid = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='neg_mean_squared_error',
    refit=True
)

grid.fit(X_train, y_train)

print("최적의 alpha:", grid.best_params_)
print("최적의 MSE:", grid.best_score_)

최적의 alpha: {'model__alpha': 1}
최적의 MSE: -12.60375165552763


## L1 Lasso Regression

Lasso는 L1 규제를 사용하는 선형회귀이다.

Ridge와 가장 큰 차이는 일부 회귀계수를 실제로 0으로 만들 수 있다는 점이다.

- 과적합을 완화하고
- 중요하지 않은 특성을 제거하는 효과도 기대할 수 있다.

따라서 Lasso는 성능뿐 아니라 몇 개의 계수가 0이 되었는지도 함께 봐야 한다.

In [46]:
lasso_result = evaluate_reg_model(
    Lasso(alpha=0.01, max_iter=50000),
    X_train, X_test, y_train, y_test
)

print("lasso_result:", lasso_result)

lasso_result: {'pipeline': Pipeline(steps=[('poly', PolynomialFeatures(include_bias=False)),
                ('scaler', StandardScaler()),
                ('model', Lasso(alpha=0.01, max_iter=50000))]), 'train_r2': 0.9158308296084051, 'test_r2': 0.8347934540883455, 'train_mse': 7.312062329714038, 'test_mse': 12.115221014551212, 'train_mae': 1.984761921126333, 'test_mae': 2.282565545139041, 'train_rmse': 2.7040825301225624, 'test_rmse': 3.4806926055817127, 'coef_l2_sum': 693.9076955470493, 'zero_coef_count': 54, 'coef_count': 104}


In [47]:
# Lasso는 Ridge보다 훨씬 작은 alpha 범위를 사용한다
# L1 규제는 조금만 커져도 계수를 빠르게 0으로 만들어버리기 때문이다
# 일반적으로 0.0001 ~ 1 사이에서 탐색한다.

lasso_alphas = [0.0001, 0.001, 0.01, 0.1, 1]

for alpha in lasso_alphas:
    result = evaluate_reg_model(
        Lasso(alpha=alpha, max_iter=50000),
        X_train, X_test, y_train, y_test
    )

    print(
        f'Lasso alpha : {alpha} '
        f'test_r2: {result['test_r2']} '
        f'test_mse: {result['test_mse']} '
        f'zero_coef: {result['zero_coef_count']}'
    )

c:\Users\Playdata\AppData\Local\miniforge3\envs\ai_basic_env\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.710e+02, tolerance: 3.510e+00
  model = cd_fast.enet_coordinate_descent(


Lasso alpha : 0.0001 test_r2: 0.811331577642222 test_mse: 13.83576917438552 zero_coef: 1


c:\Users\Playdata\AppData\Local\miniforge3\envs\ai_basic_env\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 8.151e+00, tolerance: 3.510e+00
  model = cd_fast.enet_coordinate_descent(


Lasso alpha : 0.001 test_r2: 0.8266951239663493 test_mse: 12.709102199678394 zero_coef: 17
Lasso alpha : 0.01 test_r2: 0.8347934540883455 test_mse: 12.115221014551212 zero_coef: 54
Lasso alpha : 0.1 test_r2: 0.7878811508244749 test_mse: 15.555477689655214 zero_coef: 79
Lasso alpha : 1 test_r2: 0.685565263483142 test_mse: 23.058688785800552 zero_coef: 97


## L1 + L2 ElasticNet

ElasticNet은 L1 규제와 L2 규제를 함께 사용하는 모델이다.

- Lasso는 특성 선택에 강점이 있고
- Ridge는 계수를 안정적으로 줄이는 데 강점이 있다.

ElasticNet은 이 둘의 장점을 절충하려는 방식이다.

핵심 파라미터는 다음과 같다.

1. alpha : 전체 규제 강도
2. l1_ratio : L1 비중
- 1에 가까울수록 Lasso 성향
- 0에 가까울수록 Ridge 성향

In [48]:
elastic_result = evaluate_reg_model(
    ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=10000),
    X_train, X_test, y_train, y_test
)

print("elastic result :", elastic_result)

elastic result : {'pipeline': Pipeline(steps=[('poly', PolynomialFeatures(include_bias=False)),
                ('scaler', StandardScaler()),
                ('model', ElasticNet(alpha=0.001, max_iter=10000))]), 'train_r2': 0.9308747735838176, 'test_r2': 0.8366207418554066, 'train_mse': 6.005143709497633, 'test_mse': 11.98121908967004, 'train_mae': 1.847480883266413, 'test_mae': 2.2470491456471366, 'train_rmse': 2.450539473156397, 'test_rmse': 3.4613897627499335, 'coef_l2_sum': 1490.336815023006, 'zero_coef_count': 5, 'coef_count': 104}


c:\Users\Playdata\AppData\Local\miniforge3\envs\ai_basic_env\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.065e+03, tolerance: 3.510e+00
  model = cd_fast.enet_coordinate_descent(


## 세 규제 모델 비교

같은 데이터, 같은 전처리, 같은 다항 차수에서 비교해서 해석 한다.

1. 평가셋 R²가 가장 높은 모델
2. 평가셋 MSE, MAE, RMSE가 가장 낮은 모델
3. 학습셋과 평가셋 차이가 가장 작은 모델
4. 0이 된 계수 개수가 가장 많은 모델

즉, 성능, 일반화, 모델 단순성을 함께 봐야 한다.

In [49]:
compare_df = pd.DataFrame([
    {
        'model': 'Ridge',
        'train_r2': ridge_result['train_r2'],
        'test_r2': ridge_result['test_r2'],
        'train_mse': ridge_result['train_mse'],
        'test_mse': ridge_result['test_mse'],
        'train_mae': ridge_result['train_mae'],
        'test_mae': ridge_result['test_mae'],
        'train_rmse': ridge_result['train_rmse'],
        'test_rmse': ridge_result['test_rmse'],
        'coef_l2_sum': ridge_result['coef_l2_sum'],
        'zero_coef_count': ridge_result['zero_coef_count']
    },
    {
        'model': 'Lasso',
        'train_r2': lasso_result['train_r2'],
        'test_r2': lasso_result['test_r2'],
        'train_mse': lasso_result['train_mse'],
        'test_mse': lasso_result['test_mse'],
        'train_mae': lasso_result['train_mae'],
        'test_mae': lasso_result['test_mae'],
        'train_rmse': lasso_result['train_rmse'],
        'test_rmse': lasso_result['test_rmse'],
        'coef_l2_sum': lasso_result['coef_l2_sum'],
        'zero_coef_count': lasso_result['zero_coef_count']
    },
    {
        'model': 'ElasticNet',
        'train_r2': elastic_result['train_r2'],
        'test_r2': elastic_result['test_r2'],
        'train_mse': elastic_result['train_mse'],
        'test_mse': elastic_result['test_mse'],
        'train_mae': elastic_result['train_mae'],
        'test_mae': elastic_result['test_mae'],
        'train_rmse': elastic_result['train_rmse'],
        'test_rmse': elastic_result['test_rmse'],
        'coef_l2_sum': elastic_result['coef_l2_sum'],
        'zero_coef_count': elastic_result['zero_coef_count']
    }
])

compare_df

,model,train_r2,test_r2,train_mse,test_mse,train_mae,test_mae,train_rmse,test_rmse,coef_l2_sum,zero_coef_count
0,Ridge,0.919678,0.847179,6.977833,11.206932,1.958440,2.208352,2.641559,3.347676,545.437885,0
1,Lasso,0.915831,0.834793,7.312062,12.115221,1.984762,2.282566,2.704083,3.480693,693.907696,54
2,ElasticNet,0.930875,0.836621,6.005144,11.981219,1.847481,2.247049,2.450539,3.461390,1490.336815,5


## [참고] 다중공선성 MultiCollinearity

다중공선성은 특성들끼리 서로 너무 강한 상관관계를 가지는 상태를 말한다.

예를 들어, 서로 비슷한 정보를 담는 특성이 동시에 들어오면 회귀모델은 각 변수의 역할을 안정적으로 나누기 어려워진다.

- 회귀계수가 커지거나
- 데이터가 조금만 바뀌어도 계수가 흔들리고
- 해석이 불안정해질 수 있다.

아래 예제는 실제 성능 비교용이 아니라 규제가 왜 필요한지 직관적으로 확인하기 위한 예제이다.

In [50]:
np.random.seed(42)

# 샘플 데이터 생성
X_demo = np.random.rand(100, 1)
X_demo = np.column_stack((X_demo, X_demo ** 2 + 3, X_demo ** 3 + 4))    # 서로 강하게 연관 된 특성 생성
y_demo = 3 * np.random.randn(100, 1) + np.random.randn(100, 1)

# 컬럼 간 상관 계수 확인
corr_mat = np.corrcoef(X_demo, rowvar=False)
corr_mat

array([[1.        , 0.96876011, 0.91658615],
       [0.96876011, 1.        , 0.98569849],
       [0.91658615, 0.98569849, 1.        ]])

In [51]:
# 데이터 분할
X_demo_train, X_demo_test, y_demo_train, y_demo_test = train_test_split(
    X_demo, y_demo, test_size=0.2, random_state=42
)

# 모델 비교
models = [
    Ridge(alpha=0.0),        # 사실상 규제 없는 선형 회귀와 비슷 
    Ridge(alpha=10.0),
    Lasso(alpha=0.1, max_iter=10000),
    ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=10000)
]

for model in models:
    model.fit(X_demo_train, y_demo_train)

    print(model.__class__.__name__)
    print('학습셋 MSE:', mean_squared_error(y_demo_train, model.predict(X_demo_train)))
    print('평가셋 MSE:', mean_squared_error(y_demo_test, model.predict(X_demo_test)))
    print('회귀계수:', model.coef_)
    print()

Ridge
학습셋 MSE: 7.2170159705977115
평가셋 MSE: 10.176684644740625
회귀계수: [-17.43071311  26.7371064  -11.43186193]

Ridge
학습셋 MSE: 7.818478096471177
평가셋 MSE: 11.04682273267925
회귀계수: [-0.52065163 -0.08262629  0.15334546]

Lasso
학습셋 MSE: 7.900941066505652
평가셋 MSE: 11.193791774410979
회귀계수: [-0.05683636  0.          0.        ]

ElasticNet
학습셋 MSE: 7.8413601151112235
평가셋 MSE: 11.085620414128616
회귀계수: [-0.40663041 -0.          0.        ]



## 정리

- Ridge: 모든 계수를 부드럽게 줄여 안정성을 높인다.
- Lasso: 일부 계수를 0으로 만들어 특성 선택 효과를 낸다.
- ElasticNet: 성능과 단순성 사이의 균형을 노린다.

결과를 해석할 때는 한 가지 숫자만 보면 안 된다.

1. 평가셋 R²: 설명력
2. 평가셋 MSE, MAE, RMSE: 실제 예측 오차
3. 학습/평가 차이: 과적합 여부
4. 0이 된 계수 개수: 모델 단순성

즉, 규제 모델은 단순히 성능을 높이기 위한 도구가 아니라 복잡한 모델을 더 안정적이고 해석 가능하게 만드는 도구이다.